# arch_control on a Colab GPU

Run top to bottom. Before starting, the freeze commit (`arch_control/` code + `PROTOCOL.md`) must already be **pushed to GitHub**, because this notebook clones the repository.

Results go to `OUT`. On Google Drive they survive disconnects; the runner is resumable, so after a disconnect just re-run from the *Setup* cells and then the interrupted run cell.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Results location: Google Drive if it mounts, otherwise local VM disk (download before the runtime ends).
OUT = "/content/arch_control_results"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/arch_control_results"
except Exception as e:
    print("Drive not mounted, using local disk:", e)
import os; os.makedirs(OUT, exist_ok=True); print("OUT =", OUT)

In [ ]:
REPO_URL = "https://github.com/TodManlaibaatar/Swing-by-TwoLayerReLU.git"
FREEZE_COMMIT = ""   # optional: paste the freeze commit hash to pin it exactly
import os
if not os.path.isdir("/content/Swing-by-TwoLayerReLU"):
    !git clone -q $REPO_URL /content/Swing-by-TwoLayerReLU
%cd /content/Swing-by-TwoLayerReLU
if FREEZE_COMMIT:
    !git checkout -q $FREEZE_COMMIT
!git log -1 --format='commit %H  %ad  %s'
!ls arch_control

In [ ]:
!python -c "import torch, numpy, matplotlib; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!python -m arch_control.selftest 2>&1 | grep -E "PASS|FAIL|SELFTEST"
!python -m arch_control.run --all --dry-run --out $OUT

## 2. Priority runs (headline figure, validation, step size, positive control)
The first batch prints an ETA; use it to plan the rest.

In [ ]:
!python -m arch_control.run --experiments E1_canonical E5_stepsize --device cuda --out $OUT

In [ ]:
!python -m arch_control.validate --out $OUT

In [ ]:
# E4 is tiny but has many small steps (CPU-overhead bound); cuda or cpu are similar
!python -m arch_control.run --experiments E4_positive_control --device cuda --out $OUT

In [ ]:
!python -m arch_control.analyze --out $OUT | tail -n 25

## 3. Robustness runs (grid and regime sweep; E3 is ~80% of total compute)

In [ ]:
!python -m arch_control.run --experiments E2_grid --device cuda --out $OUT

In [ ]:
!python -m arch_control.run --experiments E3_regime --device cuda --out $OUT

In [ ]:
!python -m arch_control.analyze --out $OUT | tail -n 60

## 4. Package results
`arch_control_summary.zip` (tables, figures, reports, manifests, per-run summaries) is what you commit to GitHub. `arch_control_runs.zip` holds the raw per-run arrays for the anonymous supplement.

In [ ]:
!cd $OUT && zip -qr /content/arch_control_summary.zip . -x "*/runs/*" && ls -lh /content/arch_control_summary.zip
!cd $OUT && zip -qr /content/arch_control_runs.zip . -i "*/runs/*" && ls -lh /content/arch_control_runs.zip
if OUT.startswith("/content/drive"):
    !cp /content/arch_control_summary.zip /content/arch_control_runs.zip /content/drive/MyDrive/
    print("zips copied to MyDrive")
else:
    try:
        from google.colab import files
        files.download("/content/arch_control_summary.zip")
    except Exception as e:
        print("download manually:", e)